In [ ]:
python3 -c "
   import pandas as pd
   from pathlib import Path
   import json

   parquet_file = Path('/scratch/notebook/test_ecmwf_three_stage_prebuilt_output/stage3_control_final.parquet')
   df = pd.read_parquet(parquet_file)

   print('First 20 rows:')
   print(df.head(20))
   print(f'\nTotal rows: {len(df)}')
   print(f'\nUnique key patterns (first 30):')
   keys = df['key'].tolist()[:30]
   for k in keys:
       print(f'  {k}')

In [ ]:
python3 -c "
   import pandas as pd
   import json
   from pathlib import Path

   parquet_file = Path('/scratch/notebook/test_ecmwf_three_stage_prebuilt_output/stage3_control_final.parquet')
   df = pd.read_parquet(parquet_file)

   # Find all .zattrs files and check for _ARRAY_DIMENSIONS
   print('Checking .zattrs files for _ARRAY_DIMENSIONS...')
   zattrs_files = df[df['key'].str.endswith('.zattrs', na=False)]

   missing_dims = []
   for _, row in zattrs_files.iterrows():
       key = row['key']
       value = row['value']

       if isinstance(value, bytes):
           value = value.decode('utf-8')

       try:
           attrs = json.loads(value)
           if '_ARRAY_DIMENSIONS' not in attrs:
               # Check if this corresponds to a .zarray (i.e., it's an array, not a group)
               zarray_key = key.replace('.zattrs', '.zarray')
               if zarray_key in df['key'].values:
                   missing_dims.append(key)
       except:
           pass

   print(f'\nArrays missing _ARRAY_DIMENSIONS ({len(missing_dims)}):')
   for key in sorted(missing_dims)[:30]:
       print(f'  {key}')

   # Check what _ARRAY_DIMENSIONS looks like for arrays that have it
   print('\nExample _ARRAY_DIMENSIONS values:')
   for _, row in zattrs_files.head(10).iterrows():
       key = row['key']
       value = row['value']

       if isinstance(value, bytes):
           value = value.decode('utf-8')

       try:
           attrs = json.loads(value)
           if '_ARRAY_DIMENSIONS' in attrs:
               print(f'{key}: {attrs[\"_ARRAY_DIMENSIONS\"]}')
       except:
           pass
   "

   Find arrays missing _ARRAY_DIMENSIONS

In [ ]:
python3 -c "
   import pandas as pd
   import json
   from pathlib import Path

   parquet_file = Path('/scratch/notebook/test_ecmwf_three_stage_prebuilt_output/stage3_control_final.parquet')
   df = pd.read_parquet(parquet_file)

   # Check the t2m array shape
   key = 't2m/instant/heightAboveGround/t2m/.zarray'
   row = df[df['key'] == key]
   if not row.empty:
       value = row.iloc[0]['value']
       if isinstance(value, bytes):
           value = value.decode('utf-8')
       zarray = json.loads(value)
       print(f't2m/.zarray:')
       print(f'  shape: {zarray[\"shape\"]}')
       print(f'  chunks: {zarray[\"chunks\"]}')
       print(f'  dtype: {zarray[\"dtype\"]}')

   # Check step coordinate
   key = 't2m/instant/heightAboveGround/step/.zarray'
   row = df[df['key'] == key]
   if not row.empty:
       value = row.iloc[0]['value']
       if isinstance(value, bytes):
           value = value.decode('utf-8')
       zarray = json.loads(value)
       print(f'\nstep/.zarray:')
       print(f'  shape: {zarray[\"shape\"]}')
       print(f'  chunks: {zarray[\"chunks\"]}')

   # Check what chunks exist for t2m
   print(f'\nt2m array chunks:')
   t2m_chunks = df[df['key'].str.startswith('t2m/instant/heightAboveGround/t2m/', na=False)]
   t2m_chunks = t2m_chunks[t2m_chunks['key'].str.match(r't2m/instant/heightAboveGround/t2m/\d+\.\d+\.\d+\.\d+')]
   print(f'  Total chunks: {len(t2m_chunks)}')
   for key in sorted(t2m_chunks['key'].tolist())[:10]:
       print(f'    {key}')
   "

In [ ]:
 python3 -c "
   # Test if aifs-etl.py's approach works on stage3 parquet

   import pandas as pd
   import json
   import base64
   import numpy as np

   # Read the parquet
   df = pd.read_parquet('/scratch/notebook/test_ecmwf_three_stage_prebuilt_output/stage3_control_final.parquet')

   zstore = {}
   for _, row in df.iterrows():
       key = row['key']
       value = row['value']

       if isinstance(value, bytes):
           value = value.decode('utf-8')

       if isinstance(value, str) and value.startswith('[') or value.startswith('{'):
           try:
               value = json.loads(value)
           except:
               pass

       zstore[key] = value

   if 'version' in zstore:
       del zstore['version']

   print(f'Loaded {len(zstore)} references')

   # Check the t2m/instant/heightAboveGround/t2m structure
   var_path = 't2m/instant/heightAboveGround/t2m'
   zarray_key = f'{var_path}/.zarray'

   if zarray_key in zstore:
       metadata = json.loads(zstore[zarray_key]) if isinstance(zstore[zarray_key], str) else zstore[zarray_key]
       print(f'\nVariable: {var_path}')
       print(f'  Shape: {metadata[\"shape\"]}')
       print(f'  Chunks: {metadata[\"chunks\"]}')
       print(f'  Dtype: {metadata[\"dtype\"]}')
       print(f'  Compressor: {metadata.get(\"compressor\", None)}')

       # Find all chunks
       chunks_found = []
       for key in sorted(zstore.keys()):
           if key.startswith(var_path + '/') and not key.endswith(('.zarray', '.zattrs', '.zgroup')):
               chunks_found.append(key)

       print(f'  Chunks found: {len(chunks_found)}')
       for chunk_key in chunks_found[:5]:
           print(f'    {chunk_key}')
           chunk_ref = zstore[chunk_key]
           if isinstance(chunk_ref, list):
               print(f'      Type: S3 reference')
               print(f'      URL: {chunk_ref[0][:80]}...')
           elif isinstance(chunk_ref, str) and chunk_ref.startswith('base64:'):
               print(f'      Type: base64')
           else:
               print(f'      Type: {type(chunk_ref)}')
   "


In [1]:

import pandas as pd
import re

df = pd.read_parquet('/scratch/notebook/test_ecmwf_three_stage_prebuilt_output/stage3_control_final.parquet')

# Find all step_XXX arrays for 2t variable
step_pattern = re.compile(r'step_(\d+)/2t/sfc/control/0\.0\.0')
steps = []

for key in df['key']:
   match = step_pattern.match(key)
   if match:
       step_hour = int(match.group(1))
       steps.append(step_hour)

steps = sorted(steps)
print(f'Found {len(steps)} individual timestep arrays for 2t variable')
print(f'Forecast hours: {steps[:10]}...{steps[-5:]}')
print(f'Range: {min(steps)}h to {max(steps)}h')

# Check the reference for first step
first_key = f'step_{steps[0]:03d}/2t/sfc/control/0.0.0'
first_row = df[df['key'] == first_key]
if not first_row.empty:
   import json
   ref = json.loads(first_row.iloc[0]['value'].decode('utf-8') if isinstance(first_row.iloc[0]['value'], bytes) else first_row.iloc[0]['value'])
   print(f'\nExample reference for step_{steps[0]:03d}:')
   print(f'  Type: {type(ref)}')
   if isinstance(ref, list):
       print(f'  URL: {ref[0][:80]}...')
       print(f'  Offset: {ref[1]:,}')
       print(f'  Length: {ref[2]:,}')


Found 85 individual timestep arrays for 2t variable
Forecast hours: [0, 3, 6, 9, 12, 15, 18, 21, 24, 27]...[336, 342, 348, 354, 360]
Range: 0h to 360h

Example reference for step_000:
  Type: <class 'list'>
  URL: s3://ecmwf-forecasts/20251108/00z/ifs/0p25/enfo/20251108000000-0h-enfo-ef...
  Offset: 1,634,445,168
  Length: 652,020


In [ ]:
python3 -c "
   import numpy as np

   # Load the saved data
   data = np.load('control_2t_all85.npz')

   print('✅ Successfully loaded control_2t_all85.npz')
   print(f'\nContents:')
   for key in data.files:
       if key == 'data':
           print(f'  {key}: shape={data[key].shape}, dtype={data[key].dtype}')
       elif key in ['latitude', 'longitude', 'forecast_hours']:
           arr = data[key]
           print(f'  {key}: shape={arr.shape}, range=[{arr.min():.2f}, {arr.max():.2f}]')
       else:
           print(f'  {key}: {data[key]}')

   # Show data statistics
   data_arr = data['data']
   print(f'\nData Statistics:')
   print(f'  Min: {data_arr.min():.2f}')
   print(f'  Max: {data_arr.max():.2f}')
   print(f'  Mean: {data_arr.mean():.2f}')
   print(f'  Std: {data_arr.std():.2f}')

   # Show sample values for first and last timestep
   print(f'\nSample values:')
   print(f'  First timestep (0h): {data_arr[0, 360, 720]:.2f} K')
   print(f'  Last timestep (360h): {data_arr[-1, 360, 720]:.2f} K')

   print(f'\n✅ All 85 timesteps successfully extracted and saved!')

In [ ]:
 python3 -c "
   import pandas as pd
   import re

   df = pd.read_parquet('/scratch/notebook/test_ecmwf_three_stage_prebuilt_output/stage3_ens_01_final.parquet')

   # Check what variables exist for ens_01
   print('Looking for variables in ens_01 parquet...')

   # Find all step_XXX patterns
   step_pattern = re.compile(r'step_(\d+)/([^/]+)/([^/]+)/([^/]+)/0\.0\.0')
   variables = set()
   levels = set()
   members = set()

   for key in df['key']:
       match = step_pattern.match(key)
       if match:
           step, var, level, member = match.groups()
           variables.add(var)
           levels.add(level)
           members.add(member)

   print(f'\nFound variables: {sorted(variables)[:20]}')
   print(f'Found levels: {sorted(levels)}')
   print(f'Found members: {sorted(members)}')

   # Check specifically for tp
   print(f'\nLooking for tp variable keys:')
   tp_keys = [k for k in df['key'] if 'tp' in k.lower() and 'step_' in k]
   for key in sorted(tp_keys)[:10]:
       print(f'  {key}')
   "

In [2]:
import numpy as np

# Load the saved data
data = np.load('precip.npz')

print('✅ Successfully loaded control_2t_all85.npz')
print(f'\nContents:')
for key in data.files:
   if key == 'data':
       print(f'  {key}: shape={data[key].shape}, dtype={data[key].dtype}')
   elif key in ['latitude', 'longitude', 'forecast_hours']:
       arr = data[key]
       print(f'  {key}: shape={arr.shape}, range=[{arr.min():.2f}, {arr.max():.2f}]')
   else:
       print(f'  {key}: {data[key]}')

# Show data statistics
data_arr = data['data']
print(f'\nData Statistics:')
print(f'  Min: {data_arr.min():.2f}')
print(f'  Max: {data_arr.max():.2f}')
print(f'  Mean: {data_arr.mean():.2f}')
print(f'  Std: {data_arr.std():.2f}')

# Show sample values for first and last timestep
print(f'\nSample values:')
print(f'  First timestep (0h): {data_arr[0, 360, 720]:.2f} K')
print(f'  Last timestep (360h): {data_arr[-1, 360, 720]:.2f} K')

print(f'\n✅ All 85 timesteps successfully extracted and saved!')

✅ Successfully loaded control_2t_all85.npz

Contents:
  data: shape=(85, 721, 1440), dtype=float32
  forecast_hours: shape=(85,), range=[0.00, 360.00]
  latitude: shape=(721,), range=[-90.00, 90.00]
  longitude: shape=(1440,), range=[-180.00, 179.75]
  variable: tp
  member: ens_01

Data Statistics:
  Min: 0.00
  Max: 1.35
  Mean: 0.02
  Std: 0.03

Sample values:
  First timestep (0h): 0.00 K
  Last timestep (360h): 0.06 K

✅ All 85 timesteps successfully extracted and saved!


In [ ]:
python3 -c "
   import pandas as pd
   import json
   import base64
   from pathlib import Path

   parquet_file = Path('/scratch/notebook/test_ecmwf_three_stage_prebuilt_output/stage3_control_final.parquet')
   df = pd.read_parquet(parquet_file)

   print('Searching for datetime/time metadata in parquet...\n')

   # Check for time-related keys
   time_keys = [k for k in df['key'] if 'time' in k.lower() and not k.endswith('.zarray')]
   print(f'Found {len(time_keys)} time-related keys')
   print(f'Sample keys: {time_keys[:10]}\n')

   # Try to extract reference time from time coordinate
   for time_key in ['t2m/instant/heightAboveGround/time/0',
                     'tp/accum/surface/time/0',
                     '2t/instant/surface/time/0']:
       row = df[df['key'] == time_key]
       if not row.empty:
           print(f'\n✅ Found: {time_key}')
           value = row.iloc[0]['value']

           if isinstance(value, bytes):
               value = value.decode('utf-8')

           # Try to decode
           if isinstance(value, str) and value.startswith('base64:'):
               try:
                   decoded = base64.b64decode(value[7:])
                   import struct
                   import datetime

                   # Try as int64 (seconds since epoch)
                   time_val = struct.unpack('<q', decoded)[0]
                   print(f'  Raw value: {time_val}')

                   # Convert to datetime
                   dt = datetime.datetime.utcfromtimestamp(time_val)
                   print(f'  DateTime: {dt} UTC')
                   print(f'  Model run: {dt.strftime(\"%Y-%m-%d %H:%M:%S UTC\")}')
                   break
               except Exception as e:
                   print(f'  Error decoding: {e}')

   # Also check file naming convention
   print(f'\n\nFilename-based information:')
   print(f'  Parquet file: {parquet_file.name}')

   # Check S3 URLs for datetime info
   print(f'\nChecking S3 URLs for datetime...')
   step_key = 'step_000/2t/sfc/control/0.0.0'
   row = df[df['key'] == step_key]
   if not row.empty:
       val = row.iloc[0]['value']
       if isinstance(val, bytes):
           val = val.decode('utf-8')
       ref = json.loads(val)
       url = ref[0]
       print(f'  S3 URL: {url}')

       # Parse datetime from URL
       # Format: s3://ecmwf-forecasts/YYYYMMDD/HHz/...
       import re
       match = re.search(r'/(\d{8})/(\d{2})z/', url)
       if match:
           date_str = match.group(1)
           hour_str = match.group(2)
           model_run = f'{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]} {hour_str}:00:00 UTC'
           print(f'  ✅ Model Run Time: {model_run}')